# Agent 1 — Module 3 Final Testing Notebook

This notebook tests:

- **Transcript 1**
- all other transcript files found inside `test_data`

Every transcript is processed through the current production Agent 1 pipeline.

Outputs are stored inside:

```text
tester/
└── <transcript_name>/
    ├── 01_preprocessing/
    ├── 02_chunking/
    └── 03_topics/
```

This keeps the cleaned transcript, chunking files, and topic-generation files together for every transcript.

## 1. Project setup

Keep this notebook in the `Agent_1` root folder:

```text
Agent_1/
├── app/
├── scripts/
├── test_data/
├── requirements.txt
└── Agent1_Module3_Final_Tester_Updated.ipynb
```

In [17]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

required_items = [
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "scripts",
    PROJECT_ROOT / "test_data",
]

missing_items = [
    path.name
    for path in required_items
    if not path.exists()
]

if missing_items:
    raise RuntimeError(
        "This notebook must be run from the Agent_1 root folder. "
        f"Missing: {', '.join(missing_items)}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.executable}")
print(f"Working dir  : {Path.cwd()}")

Project root : C:\Users\hp\EDTECH\Agent_1
Python       : c:\Users\hp\edtech model testing\.venv\Scripts\python.exe
Working dir  : c:\Users\hp\EDTECH\Agent_1


## 2. Verify the project virtual environment

The active Python should normally point to:

```text
Agent_1/.venv/Scripts/python.exe
```

In [18]:
expected_venv = (PROJECT_ROOT / ".venv").resolve()
active_python = Path(sys.executable).resolve()

using_project_venv = (
    expected_venv in active_python.parents
)

print(f"Expected venv       : {expected_venv}")
print(f"Active Python       : {active_python}")
print(f"Using project .venv : {using_project_venv}")

if not using_project_venv:
    print(
        "\nWARNING: Select the Agent_1 .venv kernel "
        "before running the complete notebook."
    )

Expected venv       : C:\Users\hp\EDTECH\Agent_1\.venv
Active Python       : C:\Users\hp\edtech model testing\.venv\Scripts\python.exe
Using project .venv : False



## 3. Command runner

The notebook calls your existing production scripts instead of copying Module 1, Module 2, or Module 3 code.

In [19]:
def run_module(
    module: str,
    arguments: Iterable[str] = (),
    *,
    stop_on_error: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        "-m",
        module,
        *list(arguments),
    ]

    print("\n" + "=" * 110)
    print("RUNNING:")
    print(
        " ".join(
            f'"{part}"' if " " in part else part
            for part in command
        )
    )
    print("=" * 110 + "\n")

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_lines: list[str] = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    result = subprocess.CompletedProcess(
        args=command,
        returncode=return_code,
        stdout="".join(output_lines),
        stderr=None,
    )

    print("\n" + "-" * 110)
    print(f"Exit code: {return_code}")
    print("-" * 110)

    if stop_on_error and return_code != 0:
        raise RuntimeError(
            f"Command failed with exit code {return_code}: "
            f"python -m {module}"
        )

    return result

# Part A — Run all regression tests

These tests verify that the current Module 3 improvements are still working before transcript processing begins.

In [20]:
module_3_regression_result = run_module(
    "scripts.test_module_3_regressions"
)


RUNNING:
"c:\Users\hp\edtech model testing\.venv\Scripts\python.exe" -m scripts.test_module_3_regressions

PASS: test_pure_social_chunk_is_rejected
PASS: test_ambiguous_single_word_does_not_create_bits_topic
PASS: test_incidental_integer_does_not_create_data_types_topic


KeyboardInterrupt: 

In [ ]:
module_1_2_regression_result = run_module(
    "scripts.test_module_1_2_regressions"
)


RUNNING:
"c:\Users\hp\edtech model testing\.venv\Scripts\python.exe" -m scripts.test_module_1_2_regressions

PASS: test_cross_line_sentence_deduplication
PASS: test_affirmation_tokens_are_preserved
PASS: test_notification_noise_is_removed
PASS: test_source_filename_heading_is_removed
PASS: test_synthetic_test_label_is_removed
PASS: test_genuine_lesson_introduction_is_preserved
PASS: test_page_markers_and_repeated_source_headers_are_removed
PASS: test_before_we_move_on_is_not_a_transition
PASS: test_real_transition_still_detected

ALL MODULE 1 + MODULE 2 REGRESSION TESTS PASSED

--------------------------------------------------------------------------------------------------------------
Exit code: 0
--------------------------------------------------------------------------------------------------------------


# Part B — Find Transcript 1 and all batch transcripts

The notebook searches recursively inside `test_data`.

Supported transcript formats:

```text
.docx
.pdf
.txt
```

Only files whose names contain `transcript` are selected. This prevents syllabus PDFs and unrelated documents from entering the test.

In [ ]:
TEST_DATA_DIR = PROJECT_ROOT / "test_data"

TRANSCRIPTS_TO_TEST = [
    TEST_DATA_DIR / "Transcript 1.docx",
    TEST_DATA_DIR / "Transcript_Raw_2_Networks_Cybersecurity.docx",
    TEST_DATA_DIR / "Transcript_Raw_3_Algorithms_Programming.docx",
    TEST_DATA_DIR / "Transcript_Test_1_Data_Representation.docx",
    TEST_DATA_DIR / "Transcript_Test_2_Networks_Cybersecurity.docx",
    TEST_DATA_DIR / "Transcript_Test_3_Algorithms_Programming_Stress.docx",
]

missing_files = [
    path
    for path in TRANSCRIPTS_TO_TEST
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(
        f"- {path}"
        for path in missing_files
    )
    raise FileNotFoundError(
        "These transcript files were not found:\n"
        f"{missing_text}"
    )

print(
    f"Total transcripts selected: "
    f"{len(TRANSCRIPTS_TO_TEST)}"
)

for index, path in enumerate(
    TRANSCRIPTS_TO_TEST,
    start=1,
):
    print(
        f"{index}. "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

Total transcripts selected: 6
1. test_data\Transcript 1.docx
2. test_data\Transcript_Raw_2_Networks_Cybersecurity.docx
3. test_data\Transcript_Raw_3_Algorithms_Programming.docx
4. test_data\Transcript_Test_1_Data_Representation.docx
5. test_data\Transcript_Test_2_Networks_Cybersecurity.docx
6. test_data\Transcript_Test_3_Algorithms_Programming_Stress.docx


## Optional manual selection

The automatic list above should normally be used.

To test only selected files, uncomment and edit the following cell.

In [ ]:
# TRANSCRIPTS_TO_TEST = [
#     PROJECT_ROOT / "test_data" / "Transcript 1.docx",
#     PROJECT_ROOT / "test_data" / "Transcript_Test_1_Data_Representation.docx",
#     PROJECT_ROOT / "test_data" / "Transcript_Test_2_Networks_Cybersecurity.docx",
# ]
#
# for path in TRANSCRIPTS_TO_TEST:
#     if not path.exists():
#         raise FileNotFoundError(path)

# Part C — Prepare the `tester` output folder

By default, the notebook deletes the previous `tester` folder before a fresh run.

This avoids mixing old and new outputs.

In [ ]:
TESTER_OUTPUT_ROOT = PROJECT_ROOT / "tester"

RESET_TESTER_FOLDER = True

if RESET_TESTER_FOLDER and TESTER_OUTPUT_ROOT.exists():
    shutil.rmtree(TESTER_OUTPUT_ROOT)

TESTER_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Tester output folder: {TESTER_OUTPUT_ROOT}")

Tester output folder: C:\Users\hp\EDTECH\Agent_1\tester


# Part D — Run Transcript 1 and every batch transcript

Each file is processed using:

```powershell
python -m scripts.run_agent1_pipeline --file "<path>" --output-root "tester" --run-name "<transcript_name>" --no-llm
```

The production pipeline creates the preprocessing, chunking, and topic-generation outputs.

In [ ]:
def safe_run_name(path: Path) -> str:
    name = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        path.stem,
    )
    return name.strip("._-") or "transcript"


run_records: list[dict[str, object]] = []

for index, transcript_path in enumerate(
    TRANSCRIPTS_TO_TEST,
    start=1,
):
    run_name = safe_run_name(
        transcript_path
    )

    print(
        f"\nPROCESSING {index}/{len(TRANSCRIPTS_TO_TEST)}: "
        f"{transcript_path.name}"
    )

    result = run_module(
        "scripts.run_agent1_pipeline",
        [
            "--file",
            str(transcript_path),
            "--output-root",
            str(TESTER_OUTPUT_ROOT),
            "--run-name",
            run_name,
            "--no-llm",
        ],
        stop_on_error=False,
    )

    run_records.append(
        {
            "number": index,
            "transcript": transcript_path.name,
            "source_path": str(
                transcript_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "run_name": run_name,
            "status": (
                "PASS"
                if result.returncode == 0
                else "FAIL"
            ),
            "exit_code": result.returncode,
            "output_folder": str(
                (
                    TESTER_OUTPUT_ROOT
                    / run_name
                ).relative_to(
                    PROJECT_ROOT
                )
            ),
        }
    )

run_summary_df = pd.DataFrame(
    run_records
)

display(run_summary_df)


PROCESSING 1/6: Transcript 1.docx

RUNNING:
"c:\Users\hp\edtech model testing\.venv\Scripts\python.exe" -m scripts.run_agent1_pipeline --file "C:\Users\hp\EDTECH\Agent_1\test_data\Transcript 1.docx" --output-root C:\Users\hp\EDTECH\Agent_1\tester --run-name Transcript_1 --no-llm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6815.92it/s]

AGENT 1 PIPELINE COMPLETED
Run: Transcript_1
Source: C:\Users\hp\EDTECH\Agent_1\test_data\Transcript 1.docx

Module 1 cleaned transcript:
C:\Users\hp\EDTECH\Agent_1\tester\Transcript_1\01_preprocessing\cleaned_transcript.txt

Module 2 chunks:
C:\Users\hp\EDTECH\Agent_1\tester\Transcript_1\02_chunking\chunks.json

Module 3 topics:
C:\Users\hp\EDTECH\Agent_1\tester\Transcript_1\03_topic_extraction\topics.json

Pipeline manifest:
C:\Users\hp\EDTECH\Agent_1\tester\Transcript_1\pipeline_manifest.json

--------------------------------------------------------------------------------------------------------------
Exit code: 0
---------------------

,number,transcript,source_path,run_name,status,exit_code,output_folder
0,1,Transcript 1.docx,test_data\Transcript 1.docx,Transcript_1,PASS,0,tester\Transcript_1
1,2,Transcript_Raw_2_Networks_Cybersecurity.docx,test_data\Transcript_Raw_2_Networks_Cybersecur...,Transcript_Raw_2_Networks_Cybersecurity,PASS,0,tester\Transcript_Raw_2_Networks_Cybersecurity
2,3,Transcript_Raw_3_Algorithms_Programming.docx,test_data\Transcript_Raw_3_Algorithms_Programm...,Transcript_Raw_3_Algorithms_Programming,PASS,0,tester\Transcript_Raw_3_Algorithms_Programming
3,4,Transcript_Test_1_Data_Representation.docx,test_data\Transcript_Test_1_Data_Representatio...,Transcript_Test_1_Data_Representation,PASS,0,tester\Transcript_Test_1_Data_Representation
4,5,Transcript_Test_2_Networks_Cybersecurity.docx,test_data\Transcript_Test_2_Networks_Cybersecu...,Transcript_Test_2_Networks_Cybersecurity,PASS,0,tester\Transcript_Test_2_Networks_Cybersecurity
5,6,Transcript_Test_3_Algorithms_Programming_Stres...,test_data\Transcript_Test_3_Algorithms_Program...,Transcript_Test_3_Algorithms_Programming_Stress,PASS,0,tester\Transcript_Test_3_Algorithms_Programmin...


## Stop if any transcript failed

In [ ]:
failed_runs = run_summary_df[
    run_summary_df["status"] != "PASS"
]

if not failed_runs.empty:
    display(failed_runs)
    raise RuntimeError(
        f"{len(failed_runs)} transcript run(s) failed. "
        "Review the command output above."
    )

print(
    f"ALL {len(run_summary_df)} TRANSCRIPTS "
    "PROCESSED SUCCESSFULLY"
)

ALL 6 TRANSCRIPTS PROCESSED SUCCESSFULLY


# Part E — Verify the required output files

For every transcript, the notebook checks that the following stage folders exist:

```text
01_preprocessing
02_chunking
03_topics
```

In [ ]:
REQUIRED_STAGE_FOLDERS = [
    "01_preprocessing",
    "02_chunking",
    "03_topics",
]

output_checks: list[dict[str, object]] = []

for record in run_records:
    run_folder = (
        TESTER_OUTPUT_ROOT
        / str(record["run_name"])
    )

    stage_status = {
        stage: (
            run_folder / stage
        ).is_dir()
        for stage in REQUIRED_STAGE_FOLDERS
    }

    files_by_stage = {
        stage: len(
            list(
                (
                    run_folder / stage
                ).rglob("*")
            )
        )
        if (
            run_folder / stage
        ).is_dir()
        else 0
        for stage in REQUIRED_STAGE_FOLDERS
    }

    output_checks.append(
        {
            "transcript": record[
                "transcript"
            ],
            "run_folder": str(
                run_folder.relative_to(
                    PROJECT_ROOT
                )
            ),
            "preprocessing_folder": stage_status[
                "01_preprocessing"
            ],
            "preprocessing_items": files_by_stage[
                "01_preprocessing"
            ],
            "chunking_folder": stage_status[
                "02_chunking"
            ],
            "chunking_items": files_by_stage[
                "02_chunking"
            ],
            "topics_folder": stage_status[
                "03_topics"
            ],
            "topic_items": files_by_stage[
                "03_topics"
            ],
        }
    )

output_check_df = pd.DataFrame(
    output_checks
)

display(output_check_df)

,transcript,run_folder,preprocessing_folder,preprocessing_items,chunking_folder,chunking_items,topics_folder,topic_items
0,Transcript 1.docx,tester\Transcript_1,True,5,True,2,False,0
1,Transcript_Raw_2_Networks_Cybersecurity.docx,tester\Transcript_Raw_2_Networks_Cybersecurity,True,5,True,2,False,0
2,Transcript_Raw_3_Algorithms_Programming.docx,tester\Transcript_Raw_3_Algorithms_Programming,True,5,True,2,False,0
3,Transcript_Test_1_Data_Representation.docx,tester\Transcript_Test_1_Data_Representation,True,5,True,2,False,0
4,Transcript_Test_2_Networks_Cybersecurity.docx,tester\Transcript_Test_2_Networks_Cybersecurity,True,5,True,2,False,0
5,Transcript_Test_3_Algorithms_Programming_Stres...,tester\Transcript_Test_3_Algorithms_Programmin...,True,5,True,2,False,0


# Part F — Save a tester summary

A CSV and JSON summary are saved directly inside the `tester` folder.

In [ ]:
TESTER_SUMMARY_CSV = (
    TESTER_OUTPUT_ROOT
    / "tester_summary.csv"
)

TESTER_SUMMARY_JSON = (
    TESTER_OUTPUT_ROOT
    / "tester_summary.json"
)

run_summary_df.to_csv(
    TESTER_SUMMARY_CSV,
    index=False,
)

with TESTER_SUMMARY_JSON.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_records,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved: {TESTER_SUMMARY_CSV}")
print(f"Saved: {TESTER_SUMMARY_JSON}")

Saved: C:\Users\hp\EDTECH\Agent_1\tester\tester_summary.csv
Saved: C:\Users\hp\EDTECH\Agent_1\tester\tester_summary.json


# Part G — Display outputs for a selected transcript

`SELECTED_TRANSCRIPT_INDEX = 0` displays Transcript 1 because Transcript 1 is always placed first.

In [ ]:
SELECTED_TRANSCRIPT_INDEX = 0

if not (
    0
    <= SELECTED_TRANSCRIPT_INDEX
    < len(run_records)
):
    raise IndexError(
        "SELECTED_TRANSCRIPT_INDEX is outside "
        "the available transcript range."
    )

selected_record = run_records[
    SELECTED_TRANSCRIPT_INDEX
]

selected_folder = (
    TESTER_OUTPUT_ROOT
    / str(selected_record["run_name"])
)

print(
    "Selected transcript:",
    selected_record["transcript"],
)
print(
    "Output folder:",
    selected_folder.relative_to(
        PROJECT_ROOT
    ),
)

Selected transcript: Transcript 1.docx
Output folder: tester\Transcript_1


## Display cleaned transcript

In [ ]:
cleaned_candidates = [
    selected_folder
    / "01_preprocessing"
    / "cleaned_transcript.txt",
    selected_folder
    / "01_preprocessing"
    / "deterministic_cleaned.txt",
]

cleaned_file = next(
    (
        path
        for path in cleaned_candidates
        if path.exists()
    ),
    None,
)

if cleaned_file is None:
    print(
        "No cleaned transcript text file was found."
    )
else:
    print(
        cleaned_file.relative_to(
            PROJECT_ROOT
        )
    )
    print("=" * 110)
    print(
        cleaned_file.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

tester\Transcript_1\01_preprocessing\cleaned_transcript.txt
you are you are you Yes, Alargan, here you. Okay, how's everything going on to love? Yeah, I think it's going well. How was the exam? The exam only had 100 percent. Okay, that's amazing, very well done. What the questions like, was it easy? What were the questions about? The questions, it was like we were to make a class about a poly. I believe my class. Dice at heart, a certain amount of horses and I'm yet to invite a few of them, according to. So how did you, I mean, did you make any mistakes in that or was it an easy test? Or do you think you were well prepared? What do you think? And can I play with you? Have you video on if that's okay? It's a lot easier for me to figure it out if you're around or not that way. Hello, you have me. Hello. Yeah, now I can. So now can you have me? Yeah, no, I can. Okay, just by the way, letting you know, and I hope that's all right. I'm actually recording this lesson. I might do one or two m

## Display readable chunks

In [ ]:
chunk_candidates = [
    selected_folder
    / "02_chunking"
    / "chunks_readable.txt",
    selected_folder
    / "02_chunking"
    / "chunks.txt",
]

chunk_file = next(
    (
        path
        for path in chunk_candidates
        if path.exists()
    ),
    None,
)

if chunk_file is None:
    print(
        "No readable chunking file was found."
    )
else:
    print(
        chunk_file.relative_to(
            PROJECT_ROOT
        )
    )
    print("=" * 110)
    print(
        chunk_file.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

tester\Transcript_1\02_chunking\chunks_readable.txt
AGENT 1 — MODULE 2 SEMANTIC CHUNKS

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Semantic threshold: 0.1687
Total sentences: 617
Total words: 4732
Final chunks: 11

----------------------------------------------------------------------------------------------------
CHUNK 1
Words: 500
Sentences: 72
Sentence range: 0 → 71
Boundary reason: max_size
Overlap words: 0
----------------------------------------------------------------------------------------------------
you are you are you Yes, Alargan, here you. Okay, how's everything going on to love? Yeah, I think it's going well. How was the exam? The exam only had 100 percent. Okay, that's amazing, very well done. What the questions like, was it easy? What were the questions about? The questions, it was like we were to make a class about a poly. I believe my class. Dice at heart, a certain amount of horses and I'm yet to invite a few of them, according to. So how did you, I mea

## Display readable topic generation

In [ ]:
topic_candidates = [
    selected_folder
    / "03_topics"
    / "merged_topics_readable.txt",
    selected_folder
    / "03_topics"
    / "topics_readable.txt",
    selected_folder
    / "03_topics"
    / "module_3_readable.txt",
]

topic_file = next(
    (
        path
        for path in topic_candidates
        if path.exists()
    ),
    None,
)

if topic_file is None:
    readable_topic_files = sorted(
        (
            selected_folder
            / "03_topics"
        ).glob("*readable*.txt")
    )

    topic_file = (
        readable_topic_files[0]
        if readable_topic_files
        else None
    )

if topic_file is None:
    print(
        "No readable topic-generation file was found."
    )
else:
    print(
        topic_file.relative_to(
            PROJECT_ROOT
        )
    )
    print("=" * 110)
    print(
        topic_file.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

No readable topic-generation file was found.


# Final tester-folder structure

After a successful run, the project will contain:

```text
Agent_1/
└── tester/
    ├── tester_summary.csv
    ├── tester_summary.json
    ├── Transcript_1/
    │   ├── 01_preprocessing/
    │   ├── 02_chunking/
    │   └── 03_topics/
    ├── Transcript_Test_1_Data_Representation/
    │   ├── 01_preprocessing/
    │   ├── 02_chunking/
    │   └── 03_topics/
    └── ...
```

The exact files inside each stage folder are generated by the current production pipeline, so they remain consistent with the `.py` implementation.